# 09 - Daily ETa maps

Trains a final model on all tower-days from both sites and applies it pixel by pixel to the daily
NDVI/EVI rasters from notebook 02, with the day's weather values broadcast over the scene.
The output is one 30 m ETa GeoTIFF per day, which notebook 10 summarises by crop type.

The mapping model is a random forest on seven features (the six-feature set plus NDVI mean).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
DATA_DIR = Path("../data/model_input")
NDVI_DIR = Path("../data/processed/vi_rasters_daily/NDVI")
EVI_DIR = Path("../data/processed/vi_rasters_daily/EVI")
CLIMATE_CSV = Path("../data/raw/climate/CoAgMet_Climatic_Data.csv")
OUT_DIR = Path("../results/eta_rasters")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# order matters: the model is fitted on a plain array in this column order
FEATURES = ["10-d pcp", "EVI Median", "Rs", "Tmin", "NDVI Mean", "RHmax", "Precipitation"]
TARGET = "ETa"

## Final model

Same training rows as in the evaluation notebook: both sites, 2018 excluded, complete rows only.
The in-sample R² printed below is only a sanity check of the fit; the out-of-sample skill is in notebook 08.

In [ ]:
df = pd.concat([pd.read_csv(DATA_DIR / "ASP_model_input.csv"),
                pd.read_csv(DATA_DIR / "BAU_model_input.csv")])
df["year"] = pd.to_datetime(df["date"]).dt.year
df = df[df["year"] != 2018].dropna(subset=FEATURES + [TARGET])

X, y = df[FEATURES].to_numpy(), df[TARGET].to_numpy()
rf_model = RandomForestRegressor(n_estimators=500, max_depth=None, random_state=42, n_jobs=-1)
rf_model.fit(X, y)

fit = rf_model.predict(X)
print(f"training fit: R2 = {r2_score(y, fit):.3f}, RMSE = {np.sqrt(mean_squared_error(y, fit)):.3f} mm/day")

## Predict every day

NDVI and EVI change per pixel; the weather inputs are single station values for the day.
The raster index names differ from the tower-table names (the tower table has 'NDVI Mean' and
'EVI Median' because they are footprint summaries), but for a single pixel they are the same quantity.

In [ ]:
climate = pd.read_csv(CLIMATE_CSV, parse_dates=["Date"]).set_index("Date")

n_done = 0
for ndvi_path in sorted(NDVI_DIR.glob("*.tif")):
    day = pd.to_datetime(ndvi_path.stem, format="%Y-%m-%d")
    evi_path = EVI_DIR / ndvi_path.name
    if not evi_path.exists():
        print(f"no EVI raster for {ndvi_path.stem}")
        continue
    if day not in climate.index:
        print(f"no weather data for {ndvi_path.stem}")
        continue

    with rasterio.open(ndvi_path) as src:
        ndvi = src.read(1)
        meta = src.meta.copy()
    with rasterio.open(evi_path) as src:
        evi = src.read(1)

    w = climate.loc[day]
    n_pix = ndvi.size
    X_pix = np.column_stack([
        np.full(n_pix, w["10-d pcp"]),
        evi.reshape(-1),
        np.full(n_pix, w["Rs"]),
        np.full(n_pix, w["Tmin"]),
        ndvi.reshape(-1),
        np.full(n_pix, w["RHmax"]),
        np.full(n_pix, w["Precipitation"]),
    ])
    eta = rf_model.predict(X_pix).reshape(ndvi.shape)

    meta.update(driver="GTiff", dtype="float32", count=1)
    with rasterio.open(OUT_DIR / ndvi_path.name, "w", **meta) as dst:
        dst.write(eta.astype("float32"), 1)
    n_done += 1

print(f"{n_done} daily ETa rasters written to {OUT_DIR}")